# SentraGrade: extra seeds for the ResNet50 old-vs-corrected comparison

Same code, splits and settings as `leakage_split_resnet50.ipynb`; the only change is the random seed (weight-head initialisation, data order and augmentation). Seed 42 is the main run already reported in the paper (`kaggle/results/run2_stage_matched/leakage/`); this notebook adds **seeds 43 and 44** for both protocols and all six folds, so every number can be given with a spread across three seeds. No checkpoints are saved, only the per-fold CSVs.

**Kaggle setup:** Add Input → `sentragrade-data` (with `spoilage_labels.csv`), **GPU T4**, **Internet** on, then Save Version → Save & Run All. About 3–4 h on a T4. Finished folds are written after each one, so if the session stops, re-running skips them.

In [1]:
import os, json, time, math, random, shutil, glob
from pathlib import Path
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from PIL import Image
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

# ---------------- config ----------------
SEEDS = [43, 44]                  # extra seeds; seed 42 is the main run
SEED = SEEDS[0]
CLASSES = ["Banana", "Carrot", "Guava", "Indian_Gooseberry", "Mango", "Tomato"]
PROTOCOLS = ["old", "corrected"]   # run both so the comparison uses identical code
FOLDS = list(range(6))
SPLIT_MODE = "stage_matched"       # "stage_matched" (default) or "time_blocked" (first version)
SPLIT_DIRECTION = "forward"        # "forward": train on earliest days; "reverse": train on latest days
TRAIN_FRAC = 0.60                  # share of a class's images (whole days) used for training
BUFFER_DAYS = 1                    # days dropped between train and val
EPOCHS, PATIENCE, BATCH = 25, 6, 32
LR_BACKBONE, LR_HEAD, WD = 1e-4, 1e-3, 1e-4
N_BOOT = 1000                      # session-level bootstrap resamples for AUROC CIs
SESSION_GAP_MIN = 5                # photos more than 5 min apart start a new capture session
QUICK = False                      # True = 1 epoch, 2 folds, small subsets (smoke test only)

def find_data_root():
    env = os.environ.get("SENTRAGRADE_DATA_ROOT")
    if env:
        return Path(env)
    hits = glob.glob("/kaggle/input/**/manifest_full.csv", recursive=True)
    if hits:
        return Path(hits[0]).parent
    return Path("~/sentragrade_data").expanduser()

DATA_ROOT = find_data_root()

def find_spoil_csv():
    env = os.environ.get("SPOIL_CSV")
    if env:
        return Path(env)
    hits = glob.glob("/kaggle/input/**/spoilage_labels.csv", recursive=True) + glob.glob(str(DATA_ROOT / "**/spoilage_labels.csv"), recursive=True)
    for h in hits + ["kaggle/spoilage_labels.csv", "../kaggle/spoilage_labels.csv"]:
        if os.path.exists(h):
            return Path(h)
    return None
SPOIL_CSV = find_spoil_csv()
OUT = Path("/kaggle/working/sentragrade_leakage_seeds") if Path("/kaggle/working").exists() else Path("outputs/sentragrade_leakage_seeds")
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
NUM_WORKERS = 2 if DEVICE.type == "cuda" else 0

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)
if QUICK:
    EPOCHS, FOLDS, N_BOOT = 1, [0, 3], 50
print("data root:", DATA_ROOT, "| spoilage labels:", SPOIL_CSV, "| split:", SPLIT_MODE, SPLIT_DIRECTION, "| device:", DEVICE, "| output:", OUT)
assert SPLIT_MODE == "time_blocked" or SPOIL_CSV is not None, "spoilage_labels.csv not found: add it to the dataset (needed for SPLIT_MODE='stage_matched')"

assert (DATA_ROOT / "manifest_full.csv").exists(), "manifest_full.csv not found: attach sentragrade_data.zip as a dataset"

data root: /kaggle/input/datasets/riteesh23f3002667/sentragrade-data/sentragrade_data | spoilage labels: /kaggle/input/datasets/riteesh23f3002667/sentragrade-data/spoilage_labels.csv | split: stage_matched forward | device: cuda | output: /kaggle/working/sentragrade_leakage_seeds


## 1. Build both split protocols and check them

In [2]:
man = pd.read_csv(DATA_ROOT / "manifest_full.csv", dtype={"date": str, "time": str})
man["path"] = [str(DATA_ROOT / "resized" / c / f) for c, f in zip(man.class_name, man.filename)]
missing = [p for p in man.path if not os.path.exists(p)]
assert not missing, f"{len(missing)} images missing, e.g. {missing[:3]}"
if SPOIL_CSV is not None:
    sp = pd.read_csv(SPOIL_CSV)
    man = man.drop(columns=[c for c in ["spoil_status"] if c in man.columns]).merge(sp, on=["class_name", "filename"], how="left")
    assert man.spoil_status.notna().all(), f"{man.spoil_status.isna().sum()} images have no spoilage label"
else:
    man["spoil_status"] = "unknown"

# capture sessions: a burst of ~17 photos; used as the resampling unit for the bootstrap
man["ts"] = pd.to_datetime(man.date + man.time.str.zfill(6), format="%Y%m%d%H%M%S")
man = man.sort_values(["class_name", "ts"]).reset_index(drop=True)
gap = man.groupby("class_name").ts.diff() > pd.Timedelta(minutes=SESSION_GAP_MIN)
man["session"] = man.class_name + "_" + gap.groupby(man.class_name).cumsum().astype(str)
key = dict(zip(man.class_name + "/" + man.filename, man.session))
spoil_key = dict(zip(man.class_name + "/" + man.filename, man.spoil_status))

def blocked_split(d):
    # whole capture days, in time order: train block, buffer day(s) dropped, then val and test blocks.
    # Small blocks (a stage with only 2-4 days) degrade gracefully: the buffer is skipped when it would
    # leave fewer than 2 days, and 2 days give train + test with no val.
    days = sorted(d.date.unique(), reverse=(SPLIT_DIRECTION == "reverse"))
    if len(days) == 1:
        return days, [], [], []
    if len(days) == 2:
        return days[:1], [], [], days[1:]
    n = d.groupby("date").size().reindex(days).values
    cum = np.cumsum(n) / n.sum()
    n_tr = int(np.clip((cum <= TRAIN_FRAC).sum(), 1, len(days) - 2))
    buf_n = BUFFER_DAYS if len(days) - n_tr - BUFFER_DAYS >= 2 else 0
    rest = days[n_tr + buf_n:]
    k = max(1, len(rest) // 2)
    return days[:n_tr], days[n_tr:n_tr + buf_n], rest[:k], rest[k:]

def split_class(d):
    if SPLIT_MODE == "time_blocked":
        return blocked_split(d)
    # stage-matched: every capture day is all fresh or all spoiled (checked), block each stage separately
    stage_of_day = d.groupby("date").spoil_status.agg(lambda x: x.mode().iloc[0])
    mixed = d.groupby("date").spoil_status.nunique()
    assert (mixed == 1).all(), f"{d.class_name.iloc[0]}: some days mix fresh and spoiled images"
    out = [[], [], [], []]
    for st in sorted(stage_of_day.unique()):
        part = blocked_split(d[d.date.isin(stage_of_day[stage_of_day == st].index)])
        for i in range(4):
            out[i] += list(part[i])
    return tuple(sorted(x) for x in out)

split_days, rows = {}, []
for c, d in man.groupby("class_name"):
    tr, buf, va, te = split_class(d)
    assert va and te, f"{c}: split left no val or no test days"
    split_days[c] = dict(train=tr, buffer=buf, val=va, test=te)
    fr = lambda days: f"{(d.date.isin(days) & (d.spoil_status == 'Not_spoiled')).sum()}/{(d.date.isin(days) & (d.spoil_status == 'Spoiled')).sum()}"
    rows.append({"class": c, "days": d.date.nunique(),
                 "train_days": ",".join(tr), "buffer": ",".join(buf),
                 "val_days": ",".join(va), "test_days": ",".join(te),
                 "train_n": d.date.isin(tr).sum(), "val_n": d.date.isin(va).sum(),
                 "test_n": d.date.isin(te).sum(), "dropped_n": d.date.isin(buf).sum(),
                 "train_fresh/spoiled": fr(tr), "val_fresh/spoiled": fr(va), "test_fresh/spoiled": fr(te)})
corrected_table = pd.DataFrame(rows)
display(corrected_table)
corrected_table.to_csv(OUT / "corrected_split_days.csv", index=False)

def tag(df):
    df = df.copy()
    df["path"] = [str(DATA_ROOT / "resized" / c / f) for c, f in zip(df.class_name, df.filename)]
    df["session"] = (df.class_name + "/" + df.filename).map(key)
    df["spoil_status"] = (df.class_name + "/" + df.filename).map(spoil_key)
    return df

def get_fold(protocol, k):
    held = CLASSES[k]
    known = [c for c in CLASSES if c != held]
    label_map = {c: i for i, c in enumerate(sorted(known))}
    ood = man[man.class_name == held]
    if protocol == "old":
        tr = tag(pd.read_csv(DATA_ROOT / "folds" / f"fold{k}_train.csv", dtype={"date": str}))
        va = tag(pd.read_csv(DATA_ROOT / "folds" / f"fold{k}_val.csv", dtype={"date": str}))
        te = va                    # the paper reports on val: reproduce that exactly
    else:
        kn = man[man.class_name != held]
        pick = lambda part: pd.concat([kn[(kn.class_name == c) & kn.date.isin(split_days[c][part])] for c in known])
        tr, va, te = pick("train"), pick("val"), pick("test")
        # checks: no capture day shared between blocks, buffer gap respected, test unseen
        for c in known:
            s = split_days[c]
            assert not (set(s["train"]) & set(s["val"])) and not (set(s["train"]) & set(s["test"])) and not (set(s["val"]) & set(s["test"]))
        assert set(tr.path).isdisjoint(te.path) and set(va.path).isdisjoint(te.path) and set(tr.path).isdisjoint(va.path)
    assert held not in set(tr.class_name) | set(va.class_name) | set(te.class_name)
    if QUICK:
        tr = tr.groupby("class_name").head(40); va = va.groupby("class_name").head(20)
        te = te.groupby("class_name").head(20) if protocol == "corrected" else va; ood = ood.head(60)
    return dict(train=tr, val=va, test=te, ood=ood, label_map=label_map, held=held)

summary = []
for p in PROTOCOLS:
    for k in FOLDS:
        f = get_fold(p, k)
        summary.append({"protocol": p, "fold": k, "held_out": f["held"], "train": len(f["train"]),
                        "val": len(f["val"]), "test": len(f["test"]) if p == "corrected" else "= val", "ood": len(f["ood"])})
display(pd.DataFrame(summary))

,class,days,train_days,buffer,val_days,test_days,train_n,val_n,test_n,dropped_n,train_fresh/spoiled,val_fresh/spoiled,test_fresh/spoiled
0,Banana,7,"20250911,20250915,20250916",,"20250912,20250917","20250913,20250918",119,102,102,0,51/68,51/51,51/51
1,Carrot,9,"20250725,20250728,20250729,20250730,20250731",20250801,20250802,"20250726,20250804",255,51,102,50,51/204,0/51,51/51
2,Guava,10,"20250625,20250626,20250630,20250701,20250702",20250703,"20250627,20250704","20250628,20250705",255,102,102,51,102/153,51/51,51/51
3,Indian_Gooseberry,30,"20250723,20250724,20250725,20250726,20250728,2...","20250804,20250901","20250805,20250806,20250807,20250902,20250903","20250808,20250809,20250811,20250812,20250904,2...",833,238,238,102,527/306,136/102,170/68
4,Mango,8,"20250920,20250922,20250923,20250929",,"20250924,20250930","20250926,20251003",187,103,102,0,153/34,52/51,51/51
5,Tomato,48,"20250723,20250724,20250725,20250726,20250728,2...",20250901,"20250902,20250903,20250904,20250908,20250909,2...","20250915,20250916,20250917,20250918,20250919,2...",1275,459,459,51,1207/68,408/51,408/51


,protocol,fold,held_out,train,val,test,ood
0,old,0,Banana,3978,1037,= val,323
1,old,1,Carrot,3843,1037,= val,458
2,old,2,Guava,3791,1037,= val,510
3,old,3,Indian_Gooseberry,3077,850,= val,1411
4,old,4,Mango,3892,1054,= val,392
5,old,5,Tomato,2414,680,= val,2244
6,corrected,0,Banana,2805,953,1003,323
7,corrected,1,Carrot,2669,1004,1003,458
8,corrected,2,Guava,2669,953,1003,510
9,corrected,3,Indian_Gooseberry,2091,817,867,1411


## 2. Data, model and training (same settings as the paper's ResNet50)

In [3]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class ImgDS(Dataset):
    def __init__(self, df, label_map, tf):
        self.paths = df.path.tolist()
        self.labels = [label_map.get(c, -1) for c in df.class_name]
        self.tf = tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            return self.tf(im.convert("RGB")), self.labels[i]

def loader(df, lm, tf, shuffle):
    return DataLoader(ImgDS(df, lm, tf), batch_size=BATCH, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")

class ResNetClassifier(nn.Module):
    # identical layout and weights (IMAGENET1K_V2) to the paper's run and app/server.py
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        b = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        self.stem = nn.Sequential(b.conv1, b.bn1, b.relu, b.maxpool)
        self.layer1, self.layer2, self.layer3, self.layer4 = b.layer1, b.layer2, b.layer3, b.layer4
        self.avgpool = b.avgpool
        self.fc = nn.Linear(b.fc.in_features, num_classes)
    def forward(self, x):
        x = self.layer4(self.layer3(self.layer2(self.layer1(self.stem(x)))))
        feats = torch.flatten(self.avgpool(x), 1)
        return self.fc(feats), feats

def build_model(n):
    m = ResNetClassifier(n)
    for mod in [m.stem, m.layer1, m.layer2]:   # conv1..layer2 frozen, as in the paper
        for p in mod.parameters():
            p.requires_grad = False
    return m.to(DEVICE)

@torch.no_grad()
def run_eval(model, dl):
    model.eval()
    L, Fe, Y = [], [], []
    for x, y in dl:
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            lo, fe = model(x.to(DEVICE, non_blocking=True))
        L.append(lo.float().cpu()); Fe.append(fe.float().cpu()); Y.append(y)
    return torch.cat(L), torch.cat(Fe), torch.cat(Y)

def train_fold(f):
    set_seed(SEED)
    lm = f["label_map"]; n = len(lm)
    model = build_model(n)
    counts = f["train"].class_name.map(lm).value_counts().reindex(range(n)).values
    w = torch.tensor(counts.sum() / (n * counts), dtype=torch.float32, device=DEVICE)  # inverse class frequency
    crit = nn.CrossEntropyLoss(weight=w)
    head = list(model.fc.parameters())
    back = [p for name, p in model.named_parameters() if p.requires_grad and not name.startswith("fc.")]
    opt = torch.optim.AdamW([{"params": back, "lr": LR_BACKBONE}, {"params": head, "lr": LR_HEAD}], weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")
    dl_tr = loader(f["train"], lm, train_tf, True)
    dl_va = loader(f["val"], lm, eval_tf, False)
    best, best_acc, bad, hist = None, -1.0, 0, []
    for ep in range(1, EPOCHS + 1):
        model.train(); t0 = time.time(); tl, tn = 0.0, 0
        for x, y in dl_tr:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
                loss = crit(model(x)[0], y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item() * len(y); tn += len(y)
        sched.step()
        lo, _, yv = run_eval(model, dl_va)
        vacc = (lo.argmax(1) == yv).float().mean().item()
        vloss = F.cross_entropy(lo, yv, weight=w.cpu()).item()
        hist.append({"epoch": ep, "train_loss": tl / tn, "val_loss": vloss, "val_acc": vacc})
        print(f"    ep {ep:2d}  train_loss {tl/tn:.4f}  val_loss {vloss:.4f}  val_acc {vacc:.4f}  ({time.time()-t0:.0f}s)")
        if vacc > best_acc:                     # early stopping on val accuracy, same rule as the paper's run
            best_acc, bad = vacc, 0
            best = {kk: v.detach().cpu().clone() for kk, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best)
    return model, pd.DataFrame(hist)

## 3. Scores and metrics

In [4]:
def scores_from(logits, feats, protos):
    p = logits.softmax(1)
    return {"msp": (-p.max(1).values).numpy(),
            "energy": (-torch.logsumexp(logits, 1)).numpy(),
            "proto": torch.cdist(feats, protos).min(1).values.numpy()}

def fpr_at_95(neg, pos):
    thr = np.percentile(pos, 5)            # catch 95% of held-out images
    return float((neg >= thr).mean())

def boot_auroc(neg, pos, neg_sess, pos_sess, rng):
    # resample whole capture sessions, not single photos (photos in a burst are near-copies)
    ns, ps = pd.Series(neg).groupby(neg_sess).apply(np.array), pd.Series(pos).groupby(pos_sess).apply(np.array)
    out = []
    for _ in range(N_BOOT):
        a = np.concatenate(ns.values[rng.integers(0, len(ns), len(ns))])
        b = np.concatenate(ps.values[rng.integers(0, len(ps), len(ps))])
        out.append(roc_auc_score(np.r_[np.zeros(len(a)), np.ones(len(b))], np.r_[a, b]))
    return np.percentile(out, [2.5, 97.5])

def evaluate(model, f, protocol, k):
    lm = f["label_map"]
    tr = run_eval(model, loader(f["train"], lm, eval_tf, False))
    va = run_eval(model, loader(f["val"], lm, eval_tf, False))
    te = va if protocol == "old" else run_eval(model, loader(f["test"], lm, eval_tf, False))
    od = run_eval(model, loader(f["ood"], lm, eval_tf, False))
    protos = torch.stack([tr[1][tr[2] == c].mean(0) for c in range(len(lm))])   # class means of training embeddings
    S = {name: scores_from(lo, fe, protos) for name, (lo, fe, _) in {"val": va, "test": te, "ood": od}.items()}
    rng = np.random.default_rng(SEED)
    te_df = f["val"] if protocol == "old" else f["test"]
    te_sess = te_df.session.values
    te_stage = te_df.spoil_status.values
    ood_sess = f["ood"].session.values
    rows = []
    for m in ["msp", "energy", "proto"]:
        neg, pos = S["test"][m], S["ood"][m]
        y = np.r_[np.zeros(len(neg)), np.ones(len(pos))]
        thr = np.percentile(S["val"][m], 95)                    # operating threshold set on val only
        lo_ci, hi_ci = boot_auroc(neg, pos, te_sess, ood_sess, rng)
        rows.append({"protocol": protocol, "fold": k, "held_out": f["held"], "method": m,
                     "AUROC": roc_auc_score(y, np.r_[neg, pos]), "AUROC_lo": lo_ci, "AUROC_hi": hi_ci,
                     "FPR95": fpr_at_95(neg, pos),
                     "test_false_reject_at_val_thr": float((neg > thr).mean()),
                     "test_FRR_fresh": float((neg[te_stage == "Not_spoiled"] > thr).mean()) if (te_stage == "Not_spoiled").any() else np.nan,
                     "test_FRR_spoiled": float((neg[te_stage == "Spoiled"] > thr).mean()) if (te_stage == "Spoiled").any() else np.nan,
                     "AUROC_vs_fresh_known": roc_auc_score(np.r_[np.zeros((te_stage == "Not_spoiled").sum()), np.ones(len(pos))], np.r_[neg[te_stage == "Not_spoiled"], pos]) if (te_stage == "Not_spoiled").any() else np.nan,
                     "AUROC_vs_spoiled_known": roc_auc_score(np.r_[np.zeros((te_stage == "Spoiled").sum()), np.ones(len(pos))], np.r_[neg[te_stage == "Spoiled"], pos]) if (te_stage == "Spoiled").any() else np.nan,
                     "split_mode": SPLIT_MODE if protocol != "old" else "old",
                     "ood_caught_at_val_thr": float((pos > thr).mean()),
                     "n_neg": len(neg), "n_pos": len(pos),
                     "n_neg_sessions": len(set(te_sess)), "n_pos_sessions": len(set(ood_sess))})
    acc = {"protocol": protocol, "fold": k, "held_out": f["held"],
           "val_acc": (va[0].argmax(1) == va[2]).float().mean().item(),
           "test_acc": (te[0].argmax(1) == te[2]).float().mean().item()}
    # forced labels: what the closed-set model would call each held-out image
    inv = {v: kk for kk, v in lm.items()}
    pred = od[0].argmax(1).numpy()
    forced = pd.Series([inv[i] for i in pred]).value_counts(normalize=True)
    acc["forced_label"] = f"{forced.index[0]} ({forced.iloc[0]:.1%})"
    acc["ood_mean_msp"] = od[0].softmax(1).max(1).values.mean().item()
    bundle = {"label_map": lm, "train": dict(zip(["logits", "features", "labels"], tr)),
              "val": dict(zip(["logits", "features", "labels"], va)),
              "test": dict(zip(["logits", "features", "labels"], te)),
              "ood": dict(zip(["logits", "features", "labels"], od))}
    return rows, acc, bundle

## 4. Train and evaluate every seed, protocol and fold

Results are written after each fold; re-running skips finished (seed, protocol, fold) combinations.

In [5]:
res_path, acc_path = OUT / "ood_eval_per_fold.csv", OUT / "known_accuracy.csv"
res = pd.read_csv(res_path) if res_path.exists() else pd.DataFrame()
accs = pd.read_csv(acc_path) if acc_path.exists() else pd.DataFrame()
done = set(zip(res.seed, res.protocol, res.fold)) if len(res) else set()

for SEED in SEEDS:
    for p in PROTOCOLS:
        for k in FOLDS:
            if (SEED, p, k) in done:
                print(f"skip seed {SEED} {p} fold {k} (done)"); continue
            set_seed(SEED)
            f = get_fold(p, k)
            print(f"\n=== seed {SEED} | {p} | fold {k} | held out {f['held']} ===")
            t0 = time.time()
            model, hist = train_fold(f)
            rows, acc, _ = evaluate(model, f, p, k)
            for r in rows:
                r["seed"] = SEED
            acc["seed"] = SEED
            hist.to_csv(OUT / f"history_seed{SEED}_{p}_fold{k}.csv", index=False)
            res = pd.concat([res, pd.DataFrame(rows)], ignore_index=True); res.to_csv(res_path, index=False)
            accs = pd.concat([accs, pd.DataFrame([acc])], ignore_index=True); accs.to_csv(acc_path, index=False)
            print(pd.DataFrame(rows)[["method", "AUROC", "FPR95"]].round(4).to_string(index=False))
            print(f"  test acc {acc['test_acc']:.4f} | {time.time()-t0:.0f}s")
            del model; torch.cuda.empty_cache() if DEVICE.type == "cuda" else None


=== seed 43 | old | fold 0 | held out Banana ===
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 240MB/s]
/tmp/ipykernel_23/2311386707.py:67: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")


    ep  1  train_loss 0.1366  val_loss 0.0002  val_acc 1.0000  (130s)
    ep  2  train_loss 0.0015  val_loss 0.0001  val_acc 1.0000  (17s)
    ep  3  train_loss 0.0103  val_loss 0.0001  val_acc 1.0000  (17s)
    ep  4  train_loss 0.0026  val_loss 0.0000  val_acc 1.0000  (17s)
    ep  5  train_loss 0.0002  val_loss 0.0000  val_acc 1.0000  (17s)
    ep  6  train_loss 0.0002  val_loss 0.0000  val_acc 1.0000  (17s)
    ep  7  train_loss 0.0139  val_loss 0.0046  val_acc 0.9990  (17s)
method  AUROC  FPR95
   msp 0.9987 0.0000
energy 0.9998 0.0000
 proto 0.9959 0.0154
  test acc 1.0000 | 255s

=== seed 43 | old | fold 1 | held out Carrot ===


/tmp/ipykernel_23/2311386707.py:67: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")


    ep  1  train_loss 0.1541  val_loss 0.0003  val_acc 1.0000  (16s)
    ep  2  train_loss 0.0019  val_loss 0.0017  val_acc 1.0000  (16s)
    ep  3  train_loss 0.0007  val_loss 0.0001  val_acc 1.0000  (16s)
    ep  4  train_loss 0.0006  val_loss 0.0000  val_acc 1.0000  (16s)
    ep  5  train_loss 0.0002  val_loss 0.0000  val_acc 1.0000  (16s)
    ep  6  train_loss 0.0001  val_loss 0.0000  val_acc 1.0000  (16s)
    ep  7  train_loss 0.0114  val_loss 0.0001  val_acc 1.0000  (16s)
method  AUROC  FPR95
   msp 0.9901 0.0463
energy 0.9927 0.0424
 proto 0.9965 0.0212
  test acc 1.0000 | 130s

=== seed 43 | old | fold 2 | held out Guava ===


/tmp/ipykernel_23/2311386707.py:67: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")


... (39 outputs left out)

/tmp/ipykernel_23/2311386707.py:67: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")


    ep  1  train_loss 0.1831  val_loss 0.0027  val_acc 1.0000  (11s)
    ep  2  train_loss 0.0015  val_loss 0.0156  val_acc 0.9979  (11s)
    ep  3  train_loss 0.0011  val_loss 0.0021  val_acc 1.0000  (12s)
    ep  4  train_loss 0.0026  val_loss 0.1590  val_acc 0.9832  (11s)
    ep  5  train_loss 0.0003  val_loss 0.0025  val_acc 1.0000  (12s)
    ep  6  train_loss 0.0002  val_loss 0.0010  val_acc 1.0000  (12s)
    ep  7  train_loss 0.0001  val_loss 0.0002  val_acc 1.0000  (11s)
method  AUROC  FPR95
   msp 0.9945 0.0269
energy 0.9916 0.0199
 proto 0.9633 0.0718
  test acc 0.9950 | 96s

=== seed 44 | corrected | fold 5 | held out Tomato ===


/tmp/ipykernel_23/2311386707.py:67: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")


    ep  1  train_loss 0.2868  val_loss 0.0033  val_acc 1.0000  (7s)
    ep  2  train_loss 0.0018  val_loss 0.0031  val_acc 1.0000  (7s)
    ep  3  train_loss 0.0010  val_loss 0.0013  val_acc 1.0000  (7s)
    ep  4  train_loss 0.0030  val_loss 0.0103  val_acc 0.9983  (7s)
    ep  5  train_loss 0.0031  val_loss 0.0058  val_acc 0.9983  (7s)
    ep  6  train_loss 0.0006  val_loss 0.0010  val_acc 1.0000  (7s)
    ep  7  train_loss 0.0005  val_loss 0.0004  val_acc 1.0000  (7s)
method  AUROC  FPR95
   msp 0.9219 0.2384
energy 0.9226 0.2152
 proto 0.7861 0.2523
  test acc 0.9985 | 66s


## 5. Summary across seeds

In [6]:
res = pd.read_csv(res_path); accs = pd.read_csv(acc_path)
GROUP = ["protocol", "method"]

# per seed: mean over the six folds; then mean +- sd across seeds
per_seed = (res.groupby(GROUP + ["seed"])
              .agg(AUROC=("AUROC", "mean"), FPR95=("FPR95", "mean"),
                   FRR_fresh=("test_FRR_fresh", "mean"), FRR_spoiled=("test_FRR_spoiled", "mean"),
                   AUROC_vs_spoiled=("AUROC_vs_spoiled_known", "mean"))
              .reset_index())
for c in ["FPR95", "FRR_fresh", "FRR_spoiled"]:
    per_seed[c] *= 100
display(per_seed.round(4)); per_seed.to_csv(OUT / "summary_per_seed.csv", index=False)
across = per_seed.groupby(GROUP).agg(["mean", "std"]).drop(columns="seed", level=0)
across.columns = [f"{a}_{b}" for a, b in across.columns]
print("Mean and standard deviation across seeds (each seed = mean of six folds)")
display(across.round(4)); across.to_csv(OUT / "summary_across_seeds.csv")
if {"old", "corrected"} <= set(res.protocol):
    w = per_seed.pivot_table(index=[g for g in GROUP if g != "protocol"] + ["seed"], columns="protocol", values="AUROC")
    w["drop"] = w["old"] - w["corrected"]
    drop = w.groupby(level=[i for i in range(w.index.nlevels - 1)])["drop"].agg(["mean", "std", "min", "max"])
    print("AUROC drop old -> corrected, per seed summarised")
    display(drop.round(4)); drop.to_csv(OUT / "old_vs_corrected_drop_across_seeds.csv")

display(accs.groupby(["protocol", "seed"])[["val_acc", "test_acc"]].mean().round(4))

,protocol,method,seed,AUROC,FPR95,FRR_fresh,FRR_spoiled,AUROC_vs_spoiled
0,corrected,energy,43,0.9596,14.8923,3.3731,25.4820,0.9020
1,corrected,energy,44,0.9572,14.8807,1.0832,19.3219,0.9058
2,corrected,msp,43,0.9362,22.0766,5.8302,26.2541,0.8725
3,corrected,msp,44,0.9366,19.8487,2.0607,18.1454,0.8887
4,corrected,proto,43,0.9440,12.2386,1.9622,23.0392,0.8468
5,corrected,proto,44,0.9228,13.0745,1.6243,25.0613,0.8058
6,old,energy,43,0.9762,10.4385,4.8919,4.6277,0.9760
7,old,energy,44,0.9916,2.7056,4.7532,6.8108,0.9915
8,old,msp,43,0.9676,11.6886,4.6365,5.9206,0.9664
9,old,msp,44,0.9875,4.4878,4.8046,6.5121,0.9893


Mean and standard deviation across seeds (each seed = mean of six folds)


AUROC_mean  AUROC_std  FPR95_mean  FPR95_std  \
protocol  method                                                 
corrected energy      0.9584     0.0017     14.8865     0.0082   
          msp         0.9364     0.0003     20.9627     1.5753   
          proto       0.9334     0.0150     12.6565     0.5911   
old       energy      0.9839     0.0109      6.5721     5.4680   
          msp         0.9776     0.0141      8.0882     5.0918   
          proto       0.9648     0.0117      8.4519     0.4175   

                  FRR_fresh_mean  FRR_fresh_std  FRR_spoiled_mean  \
protocol  method                                                    
corrected energy          2.2282         1.6192           22.4020   
          msp             3.9455         2.6655           22.1998   
          proto           1.7932         0.2389           24.0502   
old       energy          4.8226         0.0981            5.7192   
          msp             4.7205         0.1188            6.2163   
          proto           5.3807         0.1555            3.1345   

                  FRR_spoiled_std  AUROC_vs_spoiled_mean  AUROC_vs_spoiled_std  
protocol  method                                                                
corrected energy           4.3559                 0.9039                0.0027  
          msp              5.7337                 0.8806                0.0115  
          proto            1.4298                 0.8263                0.0290  
old       energy           1.5437                 0.9838                0.0109  
          msp              0.4183                 0.9779                0.0162  
          proto            0.2465                 0.9854                0.0047

AUROC drop old -> corrected, per seed summarised


,mean,std,min,max
method,,,,
energy,0.0255,0.0126,0.0166,0.0344
msp,0.0412,0.0138,0.0314,0.0509
proto,0.0314,0.0033,0.0290,0.0337


val_acc  test_acc
protocol  seed                   
corrected 43     0.9998    0.9930
          44     0.9984    0.9938
old       43     1.0000    1.0000
          44     1.0000    1.0000

In [7]:
shutil.make_archive(str(OUT), "zip", OUT)
print("Download:", str(OUT) + ".zip")

Download: /kaggle/working/sentragrade_leakage_seeds.zip
